# Instructor setup — Supabase Free + OpenAI
**Run on your local Jupyter before class. Never share this notebook with students.**

Downloads the **actual Kaggle dataset** (`suraj520/customer-support-ticket-dataset`) with `kagglehub`, removes direct customer identifiers, provisions database tables/roles, and embeds an **independently authored** fictional KB with OpenAI.

Works without Docker. Requires a Supabase Free project, a Supabase **Session pooler** admin URI (or another admin connection), and an OpenAI API key. Instructor seeding uses ~600 public Kaggle rows to keep Free-tier usage small. The Kaggle dataset is *synthetic*, not customer production data.

**Safety:** Do not distribute the admin URI, personal OpenAI key, or generated team passwords. Review the dataset's licensing/usage terms before sharing copies.

## Before you run: one-time setup checklist
Full details are in `README.md` → **Instructor setup**. Short version:

1. **Supabase Free project** created, DB password saved, project awake (Free projects pause after ~1 week idle).
2. **Copy the template:** in a terminal at the repo root run `cp sample.env .env`. `.env` is git-ignored. Never share it with students.
3. **Fill in `.env` (pass 1, now)**. In Supabase click **Connect** and copy:

| `.env` variable | Value | Where |
|---|---|---|
| `SUPABASE_ADMIN_DATABASE_URL` | `postgresql://postgres.<ref>:<PASSWORD>@<host>:5432/postgres` | **Connect → Session pooler** (port **5432**). URL-encode special characters in the password. |
| `LAB_PROJECT_REF` | project ref | Dashboard URL or **Project Settings → General** |
| `LAB_POOLER_HOST` | host name only | **Connect → Transaction pooler** (port 6543 is added later; copy the host exactly) |
| `OPENAI_API_KEY` | `sk-...` | https://platform.openai.com/api-keys (billing enabled) |
| `TEAM_COUNT` | e.g. `8` | Number of student groups (1–40) |
| `SUPABASE_URL` | `https://<ref>.supabase.co` | Informational only, not read by any code |

4. **Run this notebook once**, top to bottom. Missing `.env` values are asked for with a hidden prompt instead.
5. **Get the team credentials:** section E writes `instructor_team_credentials.csv` (one row per team: `team_number, schema, database_role, password`) and `instructor_app_credentials.csv` (the `lab_app_viewer` password) next to this notebook.
6. **Fill in `.env` (pass 2, afterwards)** for the Streamlit demo: `LAB_APP_DATABASE_URL` (`lab_app_viewer.<ref>` + password from the app CSV, port **6543**) and `LAB_TEAM_SCHEMAS` (`team_01,team_02,...`).
7. **Send each team only:** team number, its own password from the CSV, `LAB_PROJECT_REF`, `LAB_POOLER_HOST`, plus its own limited OpenAI key.

In [2]:
!pip install -q "kagglehub>=0.3" "pandas>=2.2,<3" "psycopg[binary]>=3.2,<4" "openai>=1.55,<3"

In [3]:
!pip install -q python-dotenv

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

## A. Instructor configuration
Copy the **session pooler (port 5432)** URI from Supabase > Connect. Never use your service-role API key as a PostgreSQL password. With special characters in DB passwords, use the exact Supabase connection URI and URL-encode reserved characters. No credential is stored in the notebook file.

In [5]:
import os, getpass, secrets, csv, re
from pathlib import Path
from contextlib import closing
import pandas as pd
import psycopg
from psycopg import sql
from openai import OpenAI
import kagglehub

ADMIN_URI = os.getenv("SUPABASE_ADMIN_DATABASE_URL") or getpass.getpass("Supabase admin session pooler URI: ")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass.getpass("Instructor OpenAI API key: ")
TEAM_COUNT = int(os.getenv("TEAM_COUNT", "8"))  # TODO INSTRUCTOR: number of student groups
DATASET_ROWS = 600
assert 1 <= TEAM_COUNT <= 40, "Keep team count manageable on Supabase Free"
admin_client = OpenAI(api_key=OPENAI_API_KEY)

def admin_conn():
    return psycopg.connect(ADMIN_URI, sslmode="require", connect_timeout=15, prepare_threshold=None)

def vstr(values):
    assert len(values)==1536
    return "[" + ",".join(str(float(v)) for v in values) + "]"

with closing(admin_conn()) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_user, version()")
        print("Database login verified:", cur.fetchone()[0])

/Users/liamnguyen/Downloads/ai_support_workshop/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Database login verified: postgres


## B. Download Kaggle CSV and inspect source
Source: https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset

The original CSV has ~8,469 tickets. We keep a small deterministic sample, remove name/email and unrelated demographics, and preserve business dimensions. `First Response Time` and `Time to Resolution` are **timestamps**. Do not treat either as a ready-made duration or claim the difference is official SLA resolution time.

In [6]:
dataset_folder = Path(kagglehub.dataset_download("suraj520/customer-support-ticket-dataset"))
files = sorted(dataset_folder.rglob("*.csv"))
assert files, "Kaggle download returned no CSV. Check network/Kaggle access."
source_file = next((p for p in files if "customer_support_tickets" in p.name.lower()), files[0])
source = pd.read_csv(source_file, low_memory=False)
print("Downloaded:", source_file.name, "source rows:", len(source))
print("Source columns:", source.columns.tolist())
assert {"Ticket ID", "Ticket Type", "Ticket Status"}.issubset(source.columns)
display(source[["Ticket ID", "Ticket Type", "Ticket Status"]].head())

100%|███████████████████████████████████████████████████████████████████████████████| 828k/828k [00:01<00:00, 723kB/s]

Extracting files...
Downloaded: customer_support_tickets.csv source rows: 8469
Source columns: ['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']


,Ticket ID,Ticket Type,Ticket Status
0,1,Technical issue,Pending Customer Response
1,2,Technical issue,Pending Customer Response
2,3,Technical issue,Closed
3,4,Billing inquiry,Closed
4,5,Billing inquiry,Closed


In [7]:
# Reduce/clean source data for the workshop. No customer name, email, age, gender.
sub = source.rename(columns={
    "Ticket ID":"ticket_id", "Ticket Type":"ticket_type",
    "Ticket Subject":"ticket_subject", "Ticket Description":"ticket_description",
    "Ticket Status":"status", "Ticket Priority":"priority",
    "Ticket Channel":"channel", "Product Purchased":"product",
    "First Response Time":"first_response_at", "Time to Resolution":"resolved_at",
    "Customer Satisfaction Rating":"satisfaction_rating", "Customer Rating":"satisfaction_rating",
    "Resolution":"resolution_text"
})
cols = ["ticket_id","ticket_type","ticket_subject","ticket_description","status",
        "priority","channel","product","first_response_at","resolved_at",
        "satisfaction_rating","resolution_text"]
for col in cols:
    if col not in sub.columns: sub[col] = None
sub = sub[cols].copy()
sub["ticket_id"] = pd.to_numeric(sub["ticket_id"],errors="coerce")
sub = sub.dropna(subset=["ticket_id"]).drop_duplicates("ticket_id")
sub = sub.sort_values("ticket_id").head(DATASET_ROWS).copy()
sub["ticket_id"] = sub["ticket_id"].astype(int)
for col in ("first_response_at","resolved_at"):
    sub[col] = pd.to_datetime(sub[col],utc=True,errors="coerce")
sub["satisfaction_rating"] = pd.to_numeric(sub["satisfaction_rating"],errors="coerce")
# Replace unresolved placeholders and prevent accidental display of emails in descriptions.
sub["ticket_description"] = sub["ticket_description"].fillna("").str.replace(
    r"\{product_purchased\}", "[product]", regex=True
).str.replace(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", "[redacted-email]",regex=True)
sub = sub.astype(object).where(pd.notna(sub), None)
print("Curated teaching rows:",len(sub),"ticket types:", sub["ticket_type"].nunique())
display(sub.head(3))

Curated teaching rows: 600 ticket types: 5


,ticket_id,ticket_type,ticket_subject,ticket_description,status,priority,channel,product,first_response_at,resolved_at,satisfaction_rating,resolution_text
0,1,Technical issue,Product setup,I'm having an issue with the [product]. Please...,Pending Customer Response,Critical,Social media,GoPro Hero,2023-06-01 12:15:36+00:00,None,None,None
1,2,Technical issue,Peripheral compatibility,I'm having an issue with the [product]. Please...,Pending Customer Response,Critical,Chat,LG Smart TV,2023-06-01 16:45:38+00:00,None,None,None
2,3,Technical issue,Network problem,I'm facing a problem with my [product]. The [p...,Closed,Low,Social media,Dell XPS,2023-06-01 11:14:38+00:00,2023-06-01 18:05:38+00:00,3.0,Case maybe show recently my computer follow.


## C. Provision instructor-owned schema and base tables
This cell makes an idempotent schema for repeated rehearsal. It doesn't grant students DDL. Existing class rows are upserted rather than duplicated.

In [8]:
DDL = [
  "CREATE SCHEMA IF NOT EXISTS extensions",
  "CREATE EXTENSION IF NOT EXISTS vector WITH SCHEMA extensions",
  "CREATE SCHEMA IF NOT EXISTS shared",
  """CREATE TABLE IF NOT EXISTS shared.ticket_types (
       ticket_type TEXT PRIMARY KEY, notes TEXT NOT NULL DEFAULT '')""",
  """CREATE TABLE IF NOT EXISTS shared.raw_tickets (
       ticket_id INTEGER PRIMARY KEY, ticket_type TEXT, ticket_subject TEXT,
       ticket_description TEXT, status TEXT, priority TEXT, channel TEXT, product TEXT,
       first_response_at TIMESTAMPTZ, resolved_at TIMESTAMPTZ,
       satisfaction_rating NUMERIC(4,2), resolution_text TEXT)""",
  """CREATE TABLE IF NOT EXISTS shared.knowledge_articles (
       article_id TEXT PRIMARY KEY, title TEXT NOT NULL, category TEXT NOT NULL,
       content TEXT NOT NULL, embedding extensions.vector(1536) NOT NULL)""",
]
with closing(admin_conn()) as conn:
    with conn.transaction():
        with conn.cursor() as cur:
            for ddl in DDL: cur.execute(ddl)
            for ticket_type in sorted(set(sub["ticket_type"].dropna())):
                cur.execute("INSERT INTO shared.ticket_types(ticket_type) VALUES (%s) ON CONFLICT DO NOTHING",(ticket_type,))
            cols = list(sub.columns)
            q = sql.SQL("INSERT INTO shared.raw_tickets ({}) VALUES ({}) ON CONFLICT (ticket_id) DO UPDATE SET {} ").format(
                sql.SQL(',').join(map(sql.Identifier,cols)),
                sql.SQL(',').join(sql.Placeholder() for _ in cols),
                sql.SQL(',').join(sql.SQL('{}=EXCLUDED.{}').format(sql.Identifier(c),sql.Identifier(c)) for c in cols if c!='ticket_id')
            )
            cur.executemany(q, [tuple(row) for row in sub.itertuples(index=False,name=None)])
print("Shared raw data and lookup ready")

Shared raw data and lookup ready


## D. Prepare knowledge articles with OpenAI embeddings
These **fictional support instructions** are independently authored for the workshop, not the Kaggle dataset's noisy `Resolution` field. Embedding once saves cost/time for every student. Rerunning this cell calls the API again.

In [9]:
ARTICLES = [('KB-001', 'Reset forgotten password', 'Technical issue', 'Use the official password-reset page and verify the account email. A reset link should be short-lived. Never request the current password.'), ('KB-002', 'Login failure after repeated attempts', 'Technical issue', 'Repeated failed sign-ins may temporarily lock an account. Verify the customer, wait for lockout expiration, and follow approved password-reset steps.'), ('KB-003', 'Device cannot connect to Wi-Fi', 'Technical issue', 'Check Wi-Fi credentials, network availability and device airplane mode. Restart device and router if permitted. Escalate with device model and error code if unresolved.'), ('KB-004', 'Product will not power on', 'Technical issue', 'Confirm power supply, cable, battery charge and manufacturer troubleshooting. Avoid instructing customers to open a device. Escalate suspected hardware defects.'), ('KB-005', 'Software installation fails', 'Technical issue', 'Verify supported operating system, available disk space, license status and installer integrity. Gather the exact error message for support escalation.'), ('KB-006', 'Duplicate billing charge', 'Billing inquiry', 'Compare transaction references and invoice dates. Escalate confirmed duplicate captures to the payments team; do not promise automatic refunds.'), ('KB-007', 'Download invoice', 'Billing inquiry', 'Billing administrators can find invoices in Billing Settings. Verify account access before sending invoice copies.'), ('KB-008', 'Payment method update', 'Billing inquiry', 'Authorized administrators can update their payment method in Billing Settings. Confirm the new method is active before removing the old one.'), ('KB-009', 'Unexpected renewal', 'Billing inquiry', 'Check renewal date, plan terms and previous cancellation requests. Escalate disputed renewals for a billing review.'), ('KB-010', 'Product compatibility', 'Product inquiry', 'Ask for the exact product model, operating system and peripheral part number. Check the official compatibility guide before recommending a purchase.'), ('KB-011', 'Product setup guide', 'Product inquiry', "Find the manufacturer's setup guide for the purchased model. Confirm software version and connection requirements before guided setup."), ('KB-012', 'Warranty coverage inquiry', 'Product inquiry', 'Verify proof of purchase, product serial number and local warranty terms. Do not assume every issue is covered.'), ('KB-013', 'Cancellation before renewal', 'Cancellation request', 'Check plan terms and the upcoming renewal date. Submit cancellation through account settings; confirm the effective date and any access implications.'), ('KB-014', 'Cancel recurring subscription', 'Cancellation request', 'Verify account authority before processing a cancellation. Explain billing-cycle effects and provide cancellation confirmation.'), ('KB-015', 'Cancel an unshipped order', 'Cancellation request', 'Review fulfillment status first. An unshipped order may be cancellable, while shipped orders use the return process.'), ('KB-016', 'Refund request requirements', 'Refund request', 'Collect transaction ID, purchase date and reason for refund. Review refund eligibility under the applicable purchase terms before approval.'), ('KB-017', 'Refund processing status', 'Refund request', 'For an approved refund, provide the refund reference and advise that bank processing times vary. Escalate missing refunds with transaction evidence.'), ('KB-018', 'Refund for defective product', 'Refund request', 'Capture device model, troubleshooting performed and proof of purchase. Evaluate warranty and return policy before promising a refund.')]
print("Articles:",len(ARTICLES))

Articles: 18


In [10]:
# Embed once via the instructor account; if rehearsing, skip by checking existing IDs first.
with closing(admin_conn()) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT article_id FROM shared.knowledge_articles")
        existing = {r[0] for r in cur.fetchall()}
needed = [a for a in ARTICLES if a[0] not in existing]
if needed:
    res = admin_client.embeddings.create(model="text-embedding-3-small",input=[f"{a[1]}\n{a[3]}" for a in needed])
    with closing(admin_conn()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                for article, item in zip(needed,res.data):
                    cur.execute("""INSERT INTO shared.knowledge_articles
                        (article_id,title,category,content,embedding)
                        VALUES (%s,%s,%s,%s,%s::extensions.vector)
                        ON CONFLICT (article_id) DO UPDATE SET title=excluded.title,category=excluded.category,
                        content=excluded.content,embedding=excluded.embedding""",
                        (*article, vstr(item.embedding)))
print("New KB embeddings:",len(needed),"/",len(ARTICLES))

New KB embeddings: 18 / 18


## E. Team schemas, least-privilege roles, and Streamlit viewer
Each group gets its own `team_NN` schema and password. App viewer can only SELECT across teams for the instructor-hosted demo. Credentials CSV is written to **your local computer only**. Do not send the entire CSV to learners.

**Run this section once.** Re-running it issues **new passwords** for every team and for `lab_app_viewer` and overwrites both CSV files, so anything already sent to students stops working.

In [12]:
roles_output=[]
with closing(admin_conn()) as conn:
    with conn.transaction():
        with conn.cursor() as cur:
            # Roles must not inherit CREATE on other schemas. Grant only targeted tables below.
            for idx in range(1,TEAM_COUNT+1):
                schema=f"team_{idx:02d}"; role=f"lab_team_{idx:02d}"
                cur.execute(sql.SQL("CREATE SCHEMA IF NOT EXISTS {}").format(sql.Identifier(schema)))
                cur.execute(sql.SQL("""CREATE TABLE IF NOT EXISTS {}.clean_tickets (
                    ticket_id INTEGER PRIMARY KEY,
                    ticket_type TEXT NOT NULL REFERENCES shared.ticket_types(ticket_type),
                    ticket_subject TEXT, status TEXT NOT NULL, priority TEXT, channel TEXT,
                    product TEXT, first_response_at TIMESTAMPTZ, resolved_at TIMESTAMPTZ,
                    satisfaction_rating NUMERIC(4,2),
                    response_to_resolution_hours NUMERIC(12,2),
                    CHECK (response_to_resolution_hours IS NULL OR response_to_resolution_hours >= 0)
                )""").format(sql.Identifier(schema)))
                cur.execute(sql.SQL("""CREATE TABLE IF NOT EXISTS {}.knowledge_articles (
                    article_id TEXT PRIMARY KEY, title TEXT NOT NULL,
                    content TEXT NOT NULL, embedding extensions.vector(1536) NOT NULL
                )""").format(sql.Identifier(schema)))
                pw=secrets.token_urlsafe(20)
                cur.execute("SELECT 1 FROM pg_roles WHERE rolname=%s",(role,))
                if cur.fetchone():
                    cur.execute(sql.SQL("ALTER ROLE {} WITH LOGIN PASSWORD {}").format(sql.Identifier(role),sql.Literal(pw)))
                else:
                    cur.execute(sql.SQL("CREATE ROLE {} WITH LOGIN PASSWORD {} NOSUPERUSER NOCREATEDB NOCREATEROLE NOREPLICATION").format(sql.Identifier(role),sql.Literal(pw)))
                cur.execute(sql.SQL("GRANT CONNECT ON DATABASE postgres TO {}").format(sql.Identifier(role)))
                for shared_schema in ("shared","extensions",schema):
                    cur.execute(sql.SQL("GRANT USAGE ON SCHEMA {} TO {}").format(sql.Identifier(shared_schema),sql.Identifier(role)))
                cur.execute(sql.SQL("GRANT SELECT ON ALL TABLES IN SCHEMA shared TO {}").format(sql.Identifier(role)))
                cur.execute(sql.SQL("GRANT SELECT,INSERT,UPDATE,DELETE ON ALL TABLES IN SCHEMA {} TO {}").format(sql.Identifier(schema),sql.Identifier(role)))
                cur.execute(sql.SQL("ALTER ROLE {} SET statement_timeout TO '25s'").format(sql.Identifier(role)))
                roles_output.append((idx,schema,role,pw))
            app_role='lab_app_viewer'; app_pw=secrets.token_urlsafe(20)
            cur.execute("SELECT 1 FROM pg_roles WHERE rolname=%s",(app_role,))
            if cur.fetchone():
                cur.execute(sql.SQL("ALTER ROLE {} WITH LOGIN PASSWORD {}").format(sql.Identifier(app_role),sql.Literal(app_pw)))
            else:
                cur.execute(sql.SQL("CREATE ROLE {} WITH LOGIN PASSWORD {} NOSUPERUSER NOCREATEDB NOCREATEROLE NOREPLICATION").format(sql.Identifier(app_role),sql.Literal(app_pw)))
            cur.execute(sql.SQL("GRANT CONNECT ON DATABASE postgres TO {}").format(sql.Identifier(app_role)))
            for sc in ['shared', 'extensions'] + [f'team_{x:02d}' for x in range(1, TEAM_COUNT + 1)]:
                cur.execute(sql.SQL("GRANT USAGE ON SCHEMA {} TO {}").format(sql.Identifier(sc),sql.Identifier(app_role)))
                cur.execute(sql.SQL("GRANT SELECT ON ALL TABLES IN SCHEMA {} TO {}").format(sql.Identifier(sc),sql.Identifier(app_role)))
            cur.execute(sql.SQL("ALTER ROLE {} SET statement_timeout TO '25s'").format(sql.Identifier(app_role)))

credfile=Path("instructor_team_credentials.csv")
with credfile.open("w",newline="") as f:
    writer=csv.writer(f);writer.writerow(['team_number','schema','database_role','password'])
    writer.writerows(roles_output)
try: credfile.chmod(0o600)
except OSError: pass
print("Provisioned",len(roles_output),"teams; credentials saved locally:",credfile.absolute())
app_credfile=Path("instructor_app_credentials.csv")
with app_credfile.open("w",newline="") as f:
    writer=csv.writer(f);writer.writerow(["database_role","password"]);writer.writerow([app_role,app_pw])
try: app_credfile.chmod(0o600)
except OSError: pass
print("Application viewer credentials saved privately to:",app_credfile.absolute())

Provisioned 4 teams; credentials saved locally: /Users/liamnguyen/Downloads/ai_support_workshop/instructor_team_credentials.csv
Application viewer credentials saved privately to: /Users/liamnguyen/Downloads/ai_support_workshop/instructor_app_credentials.csv


## F. Connectivity, class-day checklist and instructor app
Use Supabase **Transaction pooler** hostname and project reference for student logins: `lab_team_01.<project-ref>` on port `6543`. For Streamlit: `lab_app_viewer.<project-ref>` via same pooler, store its password in a private environment variable (`LAB_APP_DATABASE_URL`).

**Hand-out per team** (private message, one team each): team number, password from `instructor_team_credentials.csv`, `LAB_PROJECT_REF`, `LAB_POOLER_HOST`, own limited OpenAI key. Students enter them as Colab **Secrets** (`LAB_PROJECT_REF`, `LAB_POOLER_HOST`, `LAB_TEAM_NUMBER`, `LAB_DB_PASSWORD`, `OPENAI_API_KEY`) or in a local `.env`.

**Before class:** validate one student's credentials using a new connection, rehearse notebook 1+3, check Free project awake, and ensure OpenAI budget. **Do not upload notebook outputs containing secrets.**

In [13]:
with closing(admin_conn()) as conn:
    with conn.cursor() as cur:
        for tbl in ("shared.raw_tickets", "shared.ticket_types", "shared.knowledge_articles"):
            cur.execute(f"SELECT COUNT(*) FROM {tbl}")
            print(tbl,cur.fetchone()[0])
print("Ready: distribute only each team's username/password + pooler host/ref, not the admin URI.")

shared.raw_tickets 600
shared.ticket_types 5
shared.knowledge_articles 18
Ready: distribute only each team's username/password + pooler host/ref, not the admin URI.
